# F1 Lap Time Prediction — Data Collection

This notebook collects real Formula 1 lap data using the **FastF1** library.  
We load race sessions from the 2023 season, clean the data, and save a ready-to-use CSV for the modelling stage.

**Output:** `data/raw/laps_2023.csv`

---
## 1. Imports & Cache Setup

FastF1 downloads session data from the F1 live-timing API and caches it locally.  
Enabling the cache avoids re-downloading large files on every run — essential when iterating.

In [ ]:
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# Point the cache to data/raw/ so raw files are kept alongside the dataset
fastf1.Cache.enable_cache('../data/raw/')

print(f'FastF1 version: {fastf1.__version__}')

---
## 2. Load a Single Session (Smoke Test)

Before scaling up, we validate that the API and cache work correctly by loading  
one well-known race: the **2023 Bahrain Grand Prix**.  
This also gives us a first look at the raw data structure.

In [ ]:
# 'R' = Race session  |  other options: 'Q' Qualifying, 'FP1'/'FP2'/'FP3' Practice
session = fastf1.get_session(2023, 'Bahrain', 'R')

# Load laps + weather data (weather gets merged into the laps DataFrame automatically)
session.load(laps=True, telemetry=False, weather=True, messages=False)

print(f"Event  : {session.event['EventName']}")
print(f"Date   : {session.event['EventDate'].date()}")
print(f"Circuit: {session.event['Location']}")
print(f"Total lap rows in raw data: {len(session.laps)}")

---
## 3. Explore Lap Data

The `session.laps` object is a FastF1 `Laps` DataFrame — a pandas DataFrame with  
extra helpers. Here we inspect the raw columns to understand what's available  
before deciding what to keep.

In [ ]:
laps = session.laps

print('--- First 10 rows ---')
display(laps.head(10))

print(f'\n--- All {len(laps.columns)} available columns ---')
for col in laps.columns:
    print(f'  {col:<30} {laps[col].dtype}')

---
## 4. Filter Clean Laps Only

Raw lap data contains noise that would mislead a model:

| Problem | Why it's noise |
|---|---|
| **In-laps / Out-laps** | Driver pits mid-lap — lap time is artificially long |
| **Safety Car / VSC laps** | Lap time is dictated by the SC, not the car's pace |
| **Extreme outliers** | Spins, incidents, installation laps |

We use FastF1's `pick_quicklaps()` (keeps laps within 107 % of session fastest)  
and filter on `TrackStatus == '1'` (green flag only) to get representative laps.

We then keep only the columns relevant for ML and convert `LapTime` to seconds.

In [ ]:
# --- Remove slow laps (in/out laps, spins, etc.) ---
# pick_quicklaps keeps laps within `threshold` of the session's fastest lap
clean = laps.pick_quicklaps(threshold=1.07)

# --- Keep green-flag laps only ---
# TrackStatus '1' = clear track | '2' = yellow | '4' = SC | '6'/'7' = VSC
clean = clean[clean['TrackStatus'] == '1']

# --- Select ML-relevant columns ---
KEEP_COLS = [
    'LapTime',    # target variable
    'LapNumber',  # proxy for tyre degradation over race distance
    'Stint',      # stint number (1st, 2nd, 3rd stop)
    'TyreLife',   # laps on current tyre set
    'Compound',   # SOFT / MEDIUM / HARD / INTERMEDIATE / WET
    'AirTemp',    # °C — from merged weather data
    'Humidity',   # % relative humidity
    'TrackTemp',  # °C — stronger effect on lap time than air temp
    'WindSpeed',  # kph
    'Driver',     # driver code (HAM, VER, …)
    'Team',       # constructor name
]
clean = clean[KEEP_COLS].copy()

# --- Convert LapTime timedelta → float seconds ---
clean['LapTime'] = clean['LapTime'].dt.total_seconds()

# --- Drop rows missing critical values ---
clean = clean.dropna(subset=['LapTime', 'TyreLife', 'Compound'])

print(f'Shape after cleaning: {clean.shape}')
print(f'Drivers: {sorted(clean["Driver"].unique())}')
display(clean.head())
display(clean.describe())

---
## 5. Scale Up: Load Multiple Races

A single race (~500 clean laps) is too small to train a robust model.  
We load the first **5 rounds of the 2023 season** to get a diverse dataset  
covering different circuit types (street, high-speed, mixed) and weather conditions.

The cache ensures each session is only downloaded once — subsequent runs are instant.

In [ ]:
RACES_2023 = [
    'Bahrain',       # Round 1 — traditional circuit
    'Saudi Arabia',  # Round 2 — high-speed street circuit
    'Australia',     # Round 3 — mixed circuit
    'Azerbaijan',    # Round 4 — street circuit
    'Monaco',        # Round 6 — iconic slow street circuit
]

all_laps = []

for race in RACES_2023:
    print(f'Loading {race} GP...')
    sess = fastf1.get_session(2023, race, 'R')
    sess.load(laps=True, telemetry=False, weather=True, messages=False)

    race_laps = sess.laps.pick_quicklaps(threshold=1.07)
    race_laps = race_laps[race_laps['TrackStatus'] == '1']
    race_laps = race_laps[KEEP_COLS].copy()
    race_laps['LapTime'] = race_laps['LapTime'].dt.total_seconds()
    race_laps['GrandPrix'] = race  # label which race each lap came from

    clean_count = len(race_laps.dropna(subset=['LapTime', 'TyreLife', 'Compound']))
    print(f'  -> {clean_count} clean laps')
    all_laps.append(race_laps)

# Concatenate all races into one DataFrame
df_all = pd.concat(all_laps, ignore_index=True)
df_all = df_all.dropna(subset=['LapTime', 'TyreLife', 'Compound'])

print(f'\nTotal rows : {df_all.shape[0]}')
print(f'Columns    : {list(df_all.columns)}')
print('\nNull counts per column:')
print(df_all.isnull().sum().to_string())

print('\nRows per Grand Prix:')
print(df_all.groupby('GrandPrix').size().to_string())

---
## 6. Save the Dataset

We save the cleaned, combined DataFrame to `data/raw/laps_2023.csv`.  
This file is listed in `.gitignore` (raw data is never committed to the repo).  
Subsequent notebooks will load from this file directly — no re-fetching needed.

In [ ]:
OUTPUT_PATH = '../data/raw/laps_2023.csv'

df_all.to_csv(OUTPUT_PATH, index=False)

print(f'Dataset saved to: {OUTPUT_PATH}')
print(f'Shape           : {df_all.shape[0]} rows × {df_all.shape[1]} columns')
print(f'LapTime range   : {df_all["LapTime"].min():.2f}s – {df_all["LapTime"].max():.2f}s')
print(f'Compounds       : {sorted(df_all["Compound"].unique())}')